In [1]:
import statsmodels.api as sm
import statsmodels.formula.api as smf
import pandas as pd
import os
from physion.analysis.read_NWB import Data, scan_folder_for_NWBfiles

In [2]:
data = sm.datasets.get_rdataset("dietox", "geepack").data
print(data)

md = smf.mixedlm("Weight ~ Time", data, groups=data["Pig"])

mdf = md.fit()

print(mdf.summary())

      Pig     Evit     Cu  Litter  Start     Weight        Feed  Time
0    4601  Evit000  Cu000       1   26.5   26.50000         NaN     1
1    4601  Evit000  Cu000       1   26.5   27.59999    5.200005     2
2    4601  Evit000  Cu000       1   26.5   36.50000   17.600000     3
3    4601  Evit000  Cu000       1   26.5   40.29999   28.500000     4
4    4601  Evit000  Cu000       1   26.5   49.09998   45.200001     5
..    ...      ...    ...     ...    ...        ...         ...   ...
856  8442  Evit000  Cu175      24   25.7   73.19995   83.800003     8
857  8442  Evit000  Cu175      24   25.7   81.69995   99.800003     9
858  8442  Evit000  Cu175      24   25.7   90.29999  115.200001    10
859  8442  Evit000  Cu175      24   25.7   96.00000  133.200001    11
860  8442  Evit000  Cu175      24   25.7  103.50000  151.400002    12

[861 rows x 8 columns]
         Mixed Linear Model Regression Results
Model:            MixedLM Dependent Variable: Weight    
No. Observations: 861     Method

In [55]:
datafolder = os.path.join(os.path.expanduser('~'), 'DATA', 'In_Vivo_experiments','my_experiments','All_NWBs')
SESSIONS = scan_folder_for_NWBfiles(datafolder)
SESSIONS['nwbfiles'] = [os.path.basename(f) for f in SESSIONS['files']]
index = 8
filename = SESSIONS['files'][index]
data_ = Data(filename,
            verbose=False)



inspecting the folder "C:\Users\laura.gonzalez\DATA\In_Vivo_experiments\my_experiments\All_NWBs" [...]
 -> found n=20 datafiles (in 12.5s) 


In [62]:
data = Data(filename, verbose=False)
data.build_dFoF()
data.t_dFoF[-1]
running_dFoF_sampled = data.build_running_speed(specific_time_sampling=data.t_dFoF)
pupil_size_dFoF_sampled = data.build_pupil_diameter(specific_time_sampling=data.t_dFoF)
print(len(pupil_size_dFoF_sampled))
print(len(data.dFoF[0]))
print(len(running_dFoF_sampled))

df = pd.DataFrame()

df['dFoF'] = data.dFoF.mean(axis=0)   #average of all ROIs
df['running_speed'] = running_dFoF_sampled
df['pupil_size'] = pupil_size_dFoF_sampled

print(df)





calculating dF/F with method "percentile" [...]

  ** all ROIs passed the positive F0 criterion ** 

-> dFoF calculus done !  (calculation took 0.0s)
76723
76723
76723
           dFoF  running_speed  pupil_size
0      1.635129       0.012613    3.075410
1      1.401303       0.030105    3.078881
2      1.157409       0.043238    3.077543
3      0.974275       0.038745    3.074679
4      0.749281       0.036038    3.068822
...         ...            ...         ...
76718  0.892357       0.046686    2.218175
76719  0.740681       0.046686    2.218175
76720  0.689228       0.046686    2.218175
76721  0.738734       0.046686    2.218175
76722  0.753508       0.046686    2.218175

[76723 rows x 3 columns]


In [45]:
from physion.analysis.dataframe import NWB_to_dataframe
df = NWB_to_dataframe(filename,
                      normalize=['dFoF', 'Pupil-diameter', 'Running-Speed', 'Whisking'],
                      visual_stim_label='per-protocol-and-parameters',
                      verbose=False)
stim_keys = [k for k in df if ('VisStim' in k)]
stimID = 0*df['time']


In [52]:
print(stimID)

0        0.0
1        0.0
2        0.0
3        0.0
4        0.0
        ... 
76718    0.0
76719    0.0
76720    0.0
76721    0.0
76722    0.0
Name: time, Length: 76723, dtype: float64


In [64]:
md = smf.mixedlm("dFoF ~ running_speed", df, groups=df['pupil_size'])

#"dF ~ visual_stim + behavioral_state + visual_stimulation", df, groups=df['Pig']

mdf = md.fit()

print(mdf.summary())


         Mixed Linear Model Regression Results
Model:             MixedLM Dependent Variable: dFoF     
No. Observations:  76723   Method:             REML     
No. Groups:        76686   Scale:              0.0077   
Min. group size:   1       Log-Likelihood:     9519.8046
Max. group size:   38      Converged:          Yes      
Mean group size:   1.0                                  
--------------------------------------------------------
              Coef. Std.Err.    z    P>|z| [0.025 0.975]
--------------------------------------------------------
Intercept     0.633    0.001 785.132 0.000  0.632  0.635
running_speed 0.036    0.005   7.167 0.000  0.026  0.046
Group Var     0.038    0.121                            



In [19]:
df = pd.DataFrame()


print(data_.dFoF.mean(axis=0))
df['dFoF'] = data_.dFoF.mean(axis=0)   #average of all ROIs
df['running_speed'] = data_.running_speed(specific_time_sampling=data_.t_dFoF)

df = pd.DataFrame(data)

print(df)

[1.6351286  1.4013026  1.1574088  ... 0.6892278  0.73873425 0.75350803]


TypeError: 'numpy.ndarray' object is not callable

In [ ]:
md = smf.mixedlm("Weight ~ Time", df, groups=df["Pig"])

#"dF ~ visual_stim + behavioral_state + visual_stimulation", df, groups=df['Pig']

mdf = md.fit()

print(mdf.summary())